# Residency-7: Project-2 | NeuralNetwork&DL_R7 | Jayant Ojha

## Problem statement
In this hands-on project the goal is to build a python code for image classification from
scratch to understand the nitty gritties of building and training a model and further to
understand the advantages of neural networks. First we will implement a simple KNN
classifier and later implement a Neural Network to classify the images in the SVHN
dataset. We will compare the computational efficiency and accuracy between the
traditional methods and neural networks. 

### Dataset

there are close to 6,00,000 images in this dataset, we have extracted 60,000 images
(42000 training and 18000 test images) to do this project. The data comes in a MNIST-like format of
32-by-32 RGB images centred around a single digit (many of the images do contain some distractors
at the sides). 
https://drive.google.com/file/d/1L2-WXzguhUsCArrFUc8EEkXcj33pahoS/view


In [7]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity='all'

In [8]:
import matplotlib.pyplot as plt

from keras.layers.advanced_activations import LeakyReLU
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from tensorflow.keras import datasets, layers, models
from keras.optimizers import SGD
import h5py as h5
import os
import scipy.io
from scipy.io import loadmat
import cv2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn import metrics
import numpy as np
import tensorflow as tf
np.random.seed(42)
tf.set_random_seed(42)
tf.__version__
tf.keras.__version__
tf.reset_default_graph()

'1.14.0'

'2.2.4-tf'

*** Read data & test-train split 

In [9]:
# Open the file as readonly
h5f = h5.File('SVHN_single_grey1.h5', 'r')
# Load the training, test and validation set
X_train = h5f['X_train'][:]
y_train = h5f['y_train'][:]
X_test = h5f['X_test'][:]
y_test = h5f['y_test'][:]
X_val = h5f['X_val'][:]
y_val = h5f['y_val'][:]

In [10]:
X_train.shape

(42000, 32, 32)

In [11]:
X_test.shape

(18000, 32, 32)

In [12]:
y_train.shape

(42000,)

In [13]:
X_train.size

43008000

In [14]:
X_train = np.reshape(X_train, (-1,1024))
X_test = np.reshape(X_test, (-1,1024 ))

In [15]:
X_train.shape

(42000, 1024)

Implement and apply an optimal k-Nearest Neighbor (kNN) classifier 

In [16]:
NNH = KNeighborsClassifier(n_neighbors= 9)
NNH.fit(X_train,y_train)
ypred = NNH.predict(X_test)

KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=9, p=2,
           weights='uniform')

Print the classification metric report

In [17]:
accuracy=metrics.accuracy_score(y_test,ypred)
print("The accuracy when is {}".format(accuracy))

The accuracy when is 0.5124444444444445


In [18]:
neighbors = [2,3,7,9]
for x in neighbors:
    NNH = KNeighborsClassifier(n_neighbors= x)
    NNH.fit(X_train,y_train)
    ypred = NNH.predict(X_test)
    accuracy=metrics.accuracy_score(y_test,ypred)
    print("The accuracy when k={} is {}".format(x,accuracy))

KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=2, p=2,
           weights='uniform')

The accuracy when k=2 is 0.43644444444444447


KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=3, p=2,
           weights='uniform')

The accuracy when k=3 is 0.4617777777777778


KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=7, p=2,
           weights='uniform')

The accuracy when k=7 is 0.5070555555555556


KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=9, p=2,
           weights='uniform')

The accuracy when k=9 is 0.5124444444444445


In [19]:
 # Close this file
h5f.close()
print('Training set', X_train.shape, y_train.shape)
print('Validation set', X_val.shape, y_val.shape)
print('Test set', X_test.shape, y_test.shape)

Training set (42000, 1024) (42000,)
Validation set (60000, 32, 32) (60000,)
Test set (18000, 1024) (18000,)


In [20]:
# Open the file as readonly
h5f = h5.File('SVHN_single_grey1.h5', 'r')

# Load the training, test and validation set
X_train = h5f['X_train'][:]
y_train = h5f['y_train'][:]
X_test = h5f['X_test'][:]
y_test = h5f['y_test'][:]
X_val = h5f['X_val'][:]
y_val = h5f['y_val'][:]

In [21]:
X_train = X_train.reshape((42000, 32, 32, 1))
X_test = X_test.reshape((18000, 32, 32, 1))

In [22]:
#Early stopping with patience=5
es = EarlyStopping(monitor='val_acc', mode='max', patience=5)

Implement and apply a deep neural network classifier including and Batch Normalization for Neural Network

In [23]:
#Initialize model
model = tf.keras.models.Sequential()
#Add first convolutional layer
model.add(tf.keras.layers.Conv2D(32, #Number of filters 
                                 kernel_size=(3,3), #Size of the filter
                                 activation='relu', input_shape=(32,32,1)))
model.add(tf.keras.layers.BatchNormalization())
#Add second convolutional layer
model.add(tf.keras.layers.Conv2D(32, kernel_size=(3,3), activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
#model.output
#Flatten the output
model.add(tf.keras.layers.Flatten())
#Dense layer
model.add(tf.keras.layers.Dense(128, activation='relu'))
#Output layer
model.add(tf.keras.layers.Dense(10, activation='softmax'))
model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

W0922 12:41:49.724854 18284 deprecation.py:506] From C:\Users\ojhaj\Anaconda3\lib\site-packages\tensorflow\python\ops\init_ops.py:1251: calling VarianceScaling.__init__ (from tensorflow.python.ops.init_ops) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


In [24]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 30, 30, 32)        320       
_________________________________________________________________
batch_normalization (BatchNo (None, 30, 30, 32)        128       
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 28, 28, 32)        9248      
_________________________________________________________________
batch_normalization_1 (Batch (None, 28, 28, 32)        128       
_________________________________________________________________
flatten (Flatten)            (None, 25088)             0         
_________________________________________________________________
dense (Dense)                (None, 128)               3211392   
_________________________________________________________________
dense_1 (Dense)              (None, 10)                1

In [25]:
#Train the model
model.fit(X_train,y_train,          
          validation_data=(X_test,y_test),
          epochs=10,
          batch_size=32, 
          callbacks=[es])

Train on 42000 samples, validate on 18000 samples
Epoch 1/10
42000/42000 [==============================] - 14s 325us/sample - loss: 1.0521 - acc: 0.7495 - val_loss: 0.6279 - val_acc: 0.8136
Epoch 2/10
42000/42000 [==============================] - 12s 289us/sample - loss: 0.4168 - acc: 0.8742 - val_loss: 0.5902 - val_acc: 0.8478
Epoch 3/10
42000/42000 [==============================] - 12s 290us/sample - loss: 0.3036 - acc: 0.9068 - val_loss: 0.7465 - val_acc: 0.8086
Epoch 4/10
42000/42000 [==============================] - 12s 289us/sample - loss: 0.2336 - acc: 0.9278 - val_loss: 0.5380 - val_acc: 0.8596
Epoch 5/10
42000/42000 [==============================] - 12s 289us/sample - loss: 0.1797 - acc: 0.9444 - val_loss: 0.8032 - val_acc: 0.8234
Epoch 6/10
42000/42000 [==============================] - 12s 297us/sample - loss: 0.1471 - acc: 0.9553 - val_loss: 0.6264 - val_acc: 0.8741
Epoch 7/10
42000/42000 [==============================] - 12s 291us/sample - loss: 0.1178 - acc: 0.9632 

Understand the differences and trade-offs between traditional and NN classifiers with the help of classification metrics

In [26]:
model.evaluate(X_test,y_test)

18000/18000 [==============================] - 2s 84us/sample - loss: 0.7883 - acc: 0.8652


[0.7882852828866905, 0.86516666]

*** It is observed that the accuracy has increased from 51.24 in case of K Neighbors classifier to 86.51